In [ ]:
!pip install -q pycocotools

import os
import json
import shutil
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pycocotools import mask as cocoMask
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')

class Config:
    imgDir = '/content/drive/MyDrive/Keratoconus_Diagnosis_Project/Dataset/2-SlitLampLightbar_NEW/Images'
    annotationJson = '/content/drive/MyDrive/Keratoconus_Diagnosis_Project/Dataset/2-SlitLampLightbar_NEW/Annotations/annotations.json'

    imageSize = (256, 256)
    batchSize = 8
    numEpochs = 50
    learningRate = 1e-4
    savePath = 'vgg16_unet_cornea.h5'

cfg = Config()

def cocoPolyToMask(segmentation, imgH, imgW):
    mask = np.zeros((imgH, imgW), dtype=np.uint8)
    for seg in segmentation:
        if isinstance(seg, list):
            pts = np.array(seg).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(mask, [pts], 1)
    return mask

class CorneaDatasetGenerator(tf.keras.utils.Sequence):
    def __init__(self, imageIDs, cocoData, imgDir, batchSize=8, targetSize=(256, 256), shuffle=True):
        self.imageIDs = imageIDs
        self.imgInfo = {img['id']: img for img in cocoData['images']}
        self.annsById = {}
        for ann in cocoData['annotations']:
            self.annsById.setdefault(ann['image_id'], []).append(ann)

        self.imgDir = imgDir
        self.batchSize = batchSize
        self.targetSize = targetSize
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.imageIDs) / self.batchSize))

    def on_epoch_end(self):
        if self.shuffle:
            random.shuffle(self.imageIDs)

    def __getitem__(self, index):
        batchIDs = self.imageIDs[index * self.batchSize:(index + 1) * self.batchSize]
        X = np.zeros((len(batchIDs), *self.targetSize, 3), dtype=np.float32)
        y = np.zeros((len(batchIDs), *self.targetSize, 1), dtype=np.float32)

        for i, imgId in enumerate(batchIDs):
            info = self.imgInfo[imgId]
            imgPath = os.path.join(self.imgDir, info['file_name'])

            img = cv2.imread(imgPath)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, self.targetSize, interpolation=cv2.INTER_LANCZOS4) / 255.0

            mask = cocoPolyToMask(
                [ann['segmentation'][0] for ann in self.annsById.get(imgId, []) if 'segmentation' in ann],
                info['height'], info['width']
            )
            mask = cv2.resize(mask, self.targetSize, interpolation=cv2.INTER_LANCZOS4)

            X[i] = img
            y[i] = np.expand_dims(mask, axis=-1)
        return X, y


def VGG16_UNet(inputSize=(256, 256, 3)):
    inputs = tf.keras.Input(inputSize)
    vgg16 = VGG16(include_top=False, weights='imagenet', input_tensor=inputs)

    for layer in vgg16.layers[:15]:
        layer.trainable = False

    s1 = vgg16.get_layer('block1_conv2').output
    s2 = vgg16.get_layer('block2_conv2').output
    s3 = vgg16.get_layer('block3_conv3').output
    s4 = vgg16.get_layer('block4_conv3').output
    bridge = vgg16.get_layer('block5_conv3').output

    # Decoder
    u1 = layers.Conv2DTranspose(512, (3, 3), strides=(2, 2), padding='same')(bridge)
    u1 = layers.concatenate([u1, s4])
    u1 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u1)
    u1 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u1)

    u2 = layers.Conv2DTranspose(256, (3, 3), strides=(2, 2), padding='same')(u1)
    u2 = layers.concatenate([u2, s3])
    u2 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u2)
    u2 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u2)

    u3 = layers.Conv2DTranspose(128, (3, 3), strides=(2, 2), padding='same')(u2)
    u3 = layers.concatenate([u3, s2])
    u3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u3)
    u3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u3)

    u4 = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same')(u3)
    u4 = layers.concatenate([u4, s1])
    u4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    u4 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u4)

    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(u4)
    return tf.keras.Model(inputs, outputs)

print("Initializing VGG16-UNet pipeline...")

with open(cfg.annotationJson) as f:
    cocoData = json.load(f)

imageIds = [img['id'] for img in cocoData['images']]
trainIds, valIds = train_test_split(imageIds, test_size=0.2, random_state=42)

trainGen = CorneaDatasetGenerator(trainIds, cocoData, cfg.imgDir, batchSize=cfg.batchSize, targetSize=cfg.imageSize)
valGen = CorneaDatasetGenerator(valIds, cocoData, cfg.imgDir, batchSize=cfg.batchSize, targetSize=cfg.imageSize, shuffle=False)

model = VGG16_UNet(inputSize=(*cfg.imageSize, 3))

def combinedLoss(yTrue, yPred):
    def diceCoef(yTrue, yPred, smooth=1):
        yTrueF = tf.keras.backend.flatten(yTrue)
        yPredF = tf.keras.backend.flatten(yPred)
        intersection = tf.keras.backend.sum(yTrueF * yPredF)
        return (2. * intersection + smooth) / (tf.keras.backend.sum(yTrueF) + tf.keras.backend.sum(yPredF) + smooth)
    return (1 - diceCoef(yTrue, yPred)) + tf.keras.losses.binary_crossentropy(yTrue, yPred)

model.compile(optimizer=Adam(learning_rate=cfg.learningRate), loss=combinedLoss, metrics=['accuracy', tf.keras.metrics.IoU(num_classes=2, target_class_ids=[1])])

print("\nStarting Training...")
model.fit(trainGen, validation_data=valGen, epochs=cfg.numEpochs)

def calculateMetrics(pred, gt):
    predMask = (pred > 0.5).astype(np.uint8)
    intersection = np.logical_and(predMask, gt).sum()
    union = np.logical_or(predMask, gt).sum()
    iou = intersection / union if union != 0 else 1.0
    dice = (2. * intersection) / (predMask.sum() + gt.sum()) if (predMask.sum() + gt.sum()) != 0 else 1.0
    return iou, dice

print("EVALUATION ON VALIDATION SET")
ious, dices = [], []
for i in range(len(valGen)):
    X, y = valGen[i]
    preds = model.predict(X, verbose=0)
    for j in range(len(X)):
        iou, dice = calculateMetrics(preds[j,:,:,0], y[j,:,:,0])
        ious.append(iou)
        dices.append(dice)

print(f"Mean IoU: {np.mean(ious):.4f}")
print(f"Mean Dice: {np.mean(dices):.4f}")

Mounted at /content/drive
Initializing VGG16-UNet pipeline...
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step

Starting Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 648s 15s/step - accuracy: 0.9750 - io_u: 0.0000e+00 - loss: 0.5462 - val_accuracy: 0.9926 - val_io_u: 0.0000e+00 - val_loss: 0.1902
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 36s 927ms/step - accuracy: 0.9938 - io_u: 0.0000e+00 - loss: 0.1606 - val_accuracy: 0.9962 - val_io_u: 0.0000e+00 - val_loss: 0.0979
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 35s 881ms/step - accuracy: 0.9964 - io_u: 7.5300e-06 - loss: 0.0926 - val_accuracy: 0.9974 - val_io_u: 0.0000e+00 - val_loss: 0.0684
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 36s 896ms/step - accuracy: 0.9977 - io_u: 0.0035 - loss: 0.0589 - val_accuracy: 0.9977 - val_io_u: 0.0147 - val_loss: 0.0590
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 36s 888ms/step - accuracy: 0.9978 - io_u: 0.0294 - loss: 0.0543 - val_accuracy: 0.9982 - val_io_u: 0.0749 - val_loss: 0.0458
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 36s 890ms/step - accuracy: 0.9977 - io_u: 0.0832 - loss: 0.0577 - val_accuracy: 0.9977 - val_io_u: 0.0325 - val_loss: 0.0604

In [ ]:
model.save(cfg.savePath)
print(f"Model saved locally as {cfg.savePath}")

drive_path = '/content/drive/MyDrive/Keratoconus_Diagnosis_Project/Pre-Trained Models/SlitLamp_Lightbar-model/VGG16'
os.makedirs(drive_path, exist_ok=True)

drive_save_path = os.path.join(drive_path, 'vgg16_unet_cornea.h5')
model.save(drive_save_path)
print(f"Model also saved to Google Drive: {drive_save_path}")

Model saved locally as vgg16_unet_cornea.h5
Model also saved to Google Drive: /content/drive/MyDrive/Keratoconus_Diagnosis_Project/Pre-Trained Models/SlitLamp_Lightbar-model/VGG16/vgg16_unet_cornea.h5
